# dWGS — Part 8: Data Requirements (NEY GMS → NW GMS, RGL to SGL)

NHS England is transitioning centralised Whole Genome Sequencing (WGS) to a distributed
model (dWGS): each NHS GMS geography's **Requesting Genomic Laboratory (RGL)** submits
DNA samples directly to a **Sequencing Genomic Laboratory (SGL)**. This notebook covers
a specific subcontracted arrangement: **North East and Yorkshire GMS acting as RGL**,
submitting samples and a digital manifest to **NW GMS acting as SGL**, whose inbound
order pathway is the same NW Regional Integration Engine (RIE) → iGene ORM^O01
interface `03`-`07` already work with.

Three source documents define the requirement (all in `NotGit/`, gitignored per this
repo's convention, referenced here by name only):

- **`RGL to SGL SOP v0.4.docx`** — NHS England's SOP; Appendix 3 defines the 37-column
  national Digital Manifest CSV.
- **`dWGS Manifest CSV to HL7v2 ORM Transformation Specification v0.1.docx`** — NW GMS's
  draft spec turning that manifest into an ORM^O01 message, plus a 5-column local
  extension NEY GMS adds for NW GMS's benefit.
- **`dWGS Sample Manifest HL7v2 Data Mapping v0.1.xlsx`** — the field-by-field working
  sheet those tables were drawn from.

This is a preparation notebook: the sample data (`Input/dWGS.csv`) and the three-way
field mapping below.

## Field mapping: CSV → HL7v2 → FHIR

All 42 manifest fields (37 national, from `RGL to SGL SOP v0.4` Appendix 3, plus 5 NEY
local-extension fields from the transformation spec Section 7.2). Fields the
transformation spec marks `HL7v2 Mapping Required = FALSE` are left blank in both the
`hl7v2_field` and `fhir_field` columns — they're carried in the CSV but not transmitted
onward.

The `fhir_field` column is this notebook's own addition — neither source document
mentions FHIR. Identifiers/profiles reused from `03`-`07` are shown as-is;
`family_structure` and `participant_type` have no home in the current NW-GMSA IG and are
marked `(proposed)`, and the NGIS identifier system is marked `(TBC)` since no published
FHIR system for it was found in either source document.

**Two specimens, not one**: the `primary_sample_*` fields describe the specimen as
originally received at the GLH (blood/tissue, before extraction); `dispatched_sample_*`
describes the extracted DNA sent onward. Each is its own `SPM` segment/`Specimen`
resource - the dispatched specimen's `Specimen.parent` would reference the primary
specimen's. `primary_sample_type`'s germline/tumour distinction has no clean SPM/Specimen
field to sit in (HL7v2 has no field for it, and FHIR's closer fit is arguably an
`Observation` component, not `Specimen`) - marked low-confidence rather than guessed.

**GLH codes are not trust codes**: `ordering_entity_id` is correctly a referring NHS
Trust's ODS code (e.g. `RR8` Leeds Teaching Hospitals) — but `glh_laboratory_id` is a
*Genomic Laboratory Hub* code, a separate ODS tier above individual trusts (the same
pattern `02-work-orders-worked-example.ipynb` explains for `699X0`/`K1S6S`, North West's
own hub/site codes). Confirmed directly against the live NHS ODS API
(`directory.spineservices.nhs.uk`): North East and Yorkshire's hub is
**`699N0`, "YORKSHIRE AND NORTH EAST GLH"** (Newcastle-based) — one GLH code shared
across every referring trust in that geography, not a copy of whichever trust happens to
be ordering.

In [1]:
import pandas as pd

FIELD_MAPPING = [
    # csv_field, common_name, cardinality, field_type, hl7v2_field, fhir_field
    # cardinality/field_type source: RGL to SGL SOP v0.4 Appendix 3 (national fields) and the
    # transformation spec Section 7.2 (NEY local-extension fields) - in Input/dWGS.csv column order.
    ("referral_id", "Original Order Placer Group Number", "MUST", "String",
     "OBX-5 (OBX-3=NGIS_REFERRAL_ID)", "ServiceRequest.requisition"),
    ("clinical_indication_test_type_id", "Test Code", "OPTIONAL", "String",
     "OBR-4.1", "ServiceRequest.code.coding (England-GenomicTestDirectory)"),
    ("patient_nhs_number", "NHS Number", "OPTIONAL", "String",
     "PID-3 (NH)", "Patient.identifier (NHS number)"),
    ("patient_ngis_id", "Patient Identifier", "MUST", "String",
     "PID-3 (NGIS)", "Patient.identifier (TBC system)"),
    ("patient_date_of_birth", "Date Of Birth", "OPTIONAL", "Date",
     "PID-7.1", "Patient.birthDate"),
    ("ordering_entity_id", "Original Ordering Facility Code", "OPTIONAL", "Code (ODS Code)",
     "ORC-21", "ServiceRequest.requester (original order)"),
    ("glh_laboratory_id", "Filler Order Ordering Facility Code", "MUST", "Code (ODS Code)",
     "ORC-21", "ServiceRequest.requester (filler order)"),
    ("primary_sample_received_date", "Sample Received Date", "OPTIONAL", "Date",
     "SPM-18 (primary SPM)", "Specimen.receivedTime (primary specimen)"),
    ("primary_sample_id_as_received_by_glh", "Received Sample Identifier", "OPTIONAL", "String",
     "SPM-2.1 (primary SPM)", "Specimen.identifier (as received)"),
    ("primary_sample_id_in_glh_lims", "LIMS Sample Identifier", "OPTIONAL", "String",
     "SPM-2.2 (primary SPM)", "Specimen.identifier (GLH LIMS)"),
    ("primary_sample_type", "Sample Type", "MUST", "Code (Specimen Type SNOMED CT)",
     "SPM-11 (primary SPM, low confidence)", "Specimen.type / extension (germline vs tumour, low confidence)"),
    ("primary_sample_state", "Sample Material Type", "MUST", "String (enum)",
     "SPM-4.1 (primary SPM)", "Specimen.type (primary specimen material)"),
    ("received_sample_topography", "Sample Topography", "MUST (cancer only)", "String", "", ""),
    ("received_sample_morphology", "Sample Morphology", "OPTIONAL", "String", "", ""),
    ("received_sample_tumour_content_%", "Tumour Content", "MUST (cancer only)", "Number", "", ""),
    ("received_sample_comments", "Sample Comments", "OPTIONAL", "String", "", ""),
    ("received_sample_collection_date", "Specimen Collection Date", "OPTIONAL", "Date", "", ""),
    ("dispatched_sample_id_in_glh_lims", "Dispatched Sample Identifier", "OPTIONAL", "String", "", ""),
    ("dispatched_sample_lsid", "Specimen Barcode", "MUST", "String",
     "SPM-2.1 and OBX-5 (OBX-3=DISPATCHED_SAMPLE_LSID)", "Specimen.container.identifier"),
    ("dispatched_sample_type", "Dispatched Sample Type", "MUST", "Code (Specimen Type SNOMED CT)", "", ""),
    ("dispatched_sample_state", "Dispatched Material Type", "MUST", "String (enum)",
     "SPM-4.1 (dispatched SPM)", "Specimen.type (text-only - DNA)"),
    ("dispatched_sample_volume_(ul)", "Sample Volume", "OPTIONAL", "Number", "", ""),
    ("laboratory_remaining_volume_banked_(ul)", "Remaining Banked Volume", "OPTIONAL", "Number", "", ""),
    ("glh_concentration_(ng/ul)", "DNA Concentration", "OPTIONAL", "Number", "", ""),
    ("glh_od260/280", "DNA Purity", "OPTIONAL", "Number", "", ""),
    ("glh_din_value", "DNA Integrity Number", "OPTIONAL", "Number", "", ""),
    ("glh_percentage_DNA_over_23kb", "DNA Fragment Size", "OPTIONAL", "Number", "", ""),
    ("glh_qc_status", "QC Status", "OPTIONAL", "String", "", ""),
    ("glh_sample_dispatch_date", "Dispatch Date", "OPTIONAL", "Date", "", ""),
    ("glh_sample_consignment_number", "Consignment Number", "OPTIONAL", "String", "", ""),
    ("plating_organisation", "Plating Organisation", "OPTIONAL", "Enum", "", ""),
    ("gmc_rack_id", "Rack Identifier", "OPTIONAL", "String", "", ""),
    ("gmc_rack_well", "Rack Well Position", "OPTIONAL", "String (pattern)", "", ""),
    ("dna_extraction_protocol", "DNA Extraction Method", "OPTIONAL", "String", "", ""),
    ("prolonged_sample_storage", "Sample Storage Method", "OPTIONAL", "String", "", ""),
    ("retrospective_sample", "Retrospective Sample Flag", "OPTIONAL", "Enum", "", ""),
    ("approved_by", "Approved By", "OPTIONAL", "String", "", ""),
    ("patient_forename", "Forename", "MUST", "String", "PID-5.2", "Patient.name.given"),
    ("patient_surname", "Surname", "MUST", "String", "PID-5.1", "Patient.name.family"),
    ("family_structure", "Family Structure", "MUST", "Enumerated string",
     "OBX-5 (OBX-3=FAMILY_STRUCTURE)", "ServiceRequest.extension (proposed)"),
    ("participant_type", "Participant Type", "MUST", "Enumerated string",
     "OBX-5 (OBX-3=PARTICIPANT_TYPE)", "ServiceRequest.extension (proposed)"),
    ("clinical_information", "Clinical Information", "OPTIONAL", "String", "NTE-3", "ServiceRequest.note"),
]

field_mapping_df = pd.DataFrame(
    FIELD_MAPPING,
    columns=["csv_field", "common_name", "cardinality", "field_type", "hl7v2_field", "fhir_field"],
)
field_mapping_df

,csv_field,common_name,cardinality,field_type,hl7v2_field,fhir_field
0,referral_id,Original Order Placer Group Number,MUST,String,OBX-5 (OBX-3=NGIS_REFERRAL_ID),ServiceRequest.requisition
1,clinical_indication_test_type_id,Test Code,OPTIONAL,String,OBR-4.1,ServiceRequest.code.coding (England-GenomicTes...
2,patient_nhs_number,NHS Number,OPTIONAL,String,PID-3 (NH),Patient.identifier (NHS number)
3,patient_ngis_id,Patient Identifier,MUST,String,PID-3 (NGIS),Patient.identifier (TBC system)
4,patient_date_of_birth,Date Of Birth,OPTIONAL,Date,PID-7.1,Patient.birthDate
5,ordering_entity_id,Original Ordering Facility Code,OPTIONAL,Code (ODS Code),ORC-21,ServiceRequest.requester (original order)
6,glh_laboratory_id,Filler Order Ordering Facility Code,MUST,Code (ODS Code),ORC-21,ServiceRequest.requester (filler order)
7,primary_sample_received_date,Sample Received Date,OPTIONAL,Date,SPM-18 (primary SPM),Specimen.receivedTime (primary specimen)
8,primary_sample_id_as_received_by_glh,Received Sample Identifier,OPTIONAL,String,SPM-2.1 (primary SPM),Specimen.identifier (as received)
9,primary_sample_id_in_glh_lims,LIMS Sample Identifier,OPTIONAL,String,SPM-2.2 (primary SPM),Specimen.identifier (GLH LIMS)


## Sample manifest data

`Input/dWGS.csv`, checked against the mapping table above: every `MUST`/`MUST (cancer
only)` field is populated, and the type-sensitive fields (`Number`, `Date`,
`String (pattern)`) parse cleanly.

In [2]:
dwgs_df = pd.read_csv("Input/dWGS.csv", dtype=str, keep_default_na=False)
print(f"{len(dwgs_df)} rows, {len(dwgs_df.columns)} columns")

assert list(dwgs_df.columns) == field_mapping_df.csv_field.tolist(), \
    "Input/dWGS.csv column order must match the field mapping table above"

# Unconditional MUST fields must not be blank. "MUST (cancer only)" fields are
# conditional - this test data is all rare-disease dWGS referrals, not cancer,
# so those are correctly left blank rather than checked here.
must_fields = field_mapping_df[field_mapping_df.cardinality == "MUST"].csv_field
for field in must_fields:
    blank_rows = dwgs_df.index[dwgs_df[field].str.strip() == ""].tolist()
    assert not blank_rows, f"{field} is MUST but blank in row(s) {blank_rows}"

# Number/Date fields must parse as their stated type.
for field in field_mapping_df[field_mapping_df.field_type == "Number"].csv_field:
    dwgs_df[field].map(lambda v: float(v) if v.strip() else None)
for field in field_mapping_df[field_mapping_df.field_type == "Date"].csv_field:
    pd.to_datetime(dwgs_df[field].replace("", pd.NaT))

# gmc_rack_well must match the SOP's stated pattern.
rack_well_pattern = r"^[A-H][0][1-9]|[A-H][1][0-2]$"
assert dwgs_df.gmc_rack_well.str.match(rack_well_pattern).all(), "gmc_rack_well values must match the SOP pattern"

print("Input/dWGS.csv satisfies every cardinality/field_type constraint above.")
dwgs_df

6 rows, 42 columns
Input/dWGS.csv satisfies every cardinality/field_type constraint above.


,referral_id,clinical_indication_test_type_id,patient_nhs_number,patient_ngis_id,patient_date_of_birth,ordering_entity_id,glh_laboratory_id,primary_sample_received_date,primary_sample_id_as_received_by_glh,primary_sample_id_in_glh_lims,...,gmc_rack_well,dna_extraction_protocol,prolonged_sample_storage,retrospective_sample,approved_by,patient_forename,patient_surname,family_structure,participant_type,clinical_information
0,r2026000201,R14.1,9737383222,p2026000101,1978-01-17,RR8,699N0,2026-08-20,RR8-P0001,YNE26-P0001,...,A01,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Rob,Leeds,Singleton,Proband,Suspected inherited neurodevelopmental condition.
1,r2026000202,R59.1,9737873971,p2026000102,1947-04-27,RTR,699N0,2026-08-20,RTR-P0001,YNE26-P0002,...,A02,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Allanys,Middlesborough,Duo,Proband,
2,r2026000202,R59.1,9999999603,p2026000103,1984-11-06,RTR,699N0,2026-08-20,RTR-P0002,YNE26-P0003,...,A03,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Fourteen,Editestpatient,Duo,Family Member,
3,r2026000203,R27.3,9737383362,p2026000104,1989-07-01,RNN,699N0,2026-08-20,RNN-P0001,YNE26-P0004,...,A04,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Gilly,Brough,Trio,Proband,Patient has intellectual disability
4,r2026000203,R27.3,9999999581,p2026000105,1960-01-01,RNN,699N0,2026-08-20,RNN-P0002,YNE26-P0005,...,A05,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Thirteen,Editestpatient,Trio,Family Member,
5,r2026000203,R27.3,9999999514,p2026000106,1988-01-14,RNN,699N0,2026-08-20,RNN-P0003,YNE26-P0006,...,A06,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Six,Editestpatient,Trio,Family Member,
